# 🎯 Scoring Profile 실습

Azure AI Search의 Scoring Profile을 활용하여 검색 결과 순위에 비즈니스 로직을 적용하는 방법을 학습합니다.

## 📋 학습 내용

1. **Scoring Profile 개념**: 검색 점수에 비즈니스 로직 추가
2. **필드 가중치 (Text Weights)**: 필드별 중요도 설정
3. **Magnitude 함수**: 가격 기반 부스팅
4. **Tag 함수**: 특정 브랜드 우선 노출
5. **실무 시나리오**: 카테고리별 맞춤 검색

## 💡 주요 개념

```
최종 점수 = BM25 기본 점수 × 필드 가중치 × 함수 부스팅
```

---
## 1️⃣ 환경 설정 및 클라이언트 초기화

In [ ]:
import os
from dotenv import load_dotenv
from azure.core.credentials import AzureKeyCredential
from azure.search.documents import SearchClient
from azure.search.documents.indexes import SearchIndexClient
from azure.search.documents.indexes.models import (
    ScoringProfile,
    TextWeights,
    MagnitudeScoringFunction,
    MagnitudeScoringParameters,
    TagScoringFunction,
    TagScoringParameters,
    ScoringFunctionInterpolation,
)

# 환경 변수 로드
load_dotenv(override=True)

# Azure AI Search 설정
SEARCH_ENDPOINT = os.getenv("SEARCH_ENDPOINT")
SEARCH_ADMIN_KEY = os.getenv("SEARCH_ADMIN_KEY")
INDEX_NAME = os.getenv("SEARCH_INDEX_NAME", "products-index")

credential = AzureKeyCredential(SEARCH_ADMIN_KEY)

# 클라이언트 초기화
index_client = SearchIndexClient(endpoint=SEARCH_ENDPOINT, credential=credential)
search_client = SearchClient(endpoint=SEARCH_ENDPOINT, index_name=INDEX_NAME, credential=credential)

print(f"✅ Endpoint: {SEARCH_ENDPOINT}")
print(f"✅ Index: {INDEX_NAME}")
print(f"✅ 클라이언트 초기화 완료")

---
## 2️⃣ Scoring Profile 개념 이해

### Scoring Profile이란?

**Scoring Profile**은 검색 결과의 순위를 비즈니스 요구사항에 맞게 조정하는 기능입니다.

### 주요 구성 요소

| 구성 요소 | 설명 | 적용 예시 |
|----------|------|----------|
| **Text Weights** | 필드별 가중치 | title 매칭 시 2배 점수 |
| **Magnitude** | 숫자 필드 기반 | 저가 상품 우선 |
| **Tag** | 특정 값 매칭 | 프리미엄 브랜드 우선 |
| **Freshness** | 날짜 기반 | 최신 상품 우선 |
| **Distance** | 위치 기반 | 가까운 매장 우선 |

---
## 3️⃣ 기존 인덱스 확인 및 Scoring Profile 정의

In [ ]:
# 기존 인덱스 확인
existing_index = index_client.get_index(INDEX_NAME)
print(f"\n📊 현재 인덱스 정보")
print(f"   - 인덱스명: {existing_index.name}")
print(f"   - 필드 수: {len(existing_index.fields)}")
print(f"   - 필드 목록: {', '.join([f.name for f in existing_index.fields])}")
if existing_index.scoring_profiles:
    print(f"   - 기존 Scoring Profiles: {[p.name for p in existing_index.scoring_profiles]}")
else:
    print(f"   - 기존 Scoring Profiles: 없음")

In [ ]:
# Scoring Profile 정의
print("\n🔧 새로운 Scoring Profile 정의...\n")

# 프로파일 1: 필드 가중치 (title 매칭 우선)
profile_field_weights = ScoringProfile(
    name="titleBoost",
    text_weights=TextWeights(
        weights={
            "title": 3.0,      # title 필드 매칭 시 3배
            "brand": 2.0,      # brand 필드 매칭 시 2배
            "review": 1.0      # review 필드는 기본
        }
    )
)
print("   1️⃣  titleBoost: 필드 가중치")
print("       - title 필드 매칭 시 3배 부스팅")
print("       - brand 필드 매칭 시 2배 부스팅")

# 프로파일 2: 저가 상품 우선 (Magnitude 함수)
profile_low_price = ScoringProfile(
    name="lowPriceFirst",
    functions=[
        MagnitudeScoringFunction(
            field_name="normal_price",
            boost=5,
            parameters=MagnitudeScoringParameters(
                boosting_range_start=0,
                boosting_range_end=100000,
                constant_boost_beyond_range=False
            ),
            interpolation=ScoringFunctionInterpolation.LINEAR
        )
    ]
)
print("\n   2️⃣  lowPriceFirst: 저가 상품 우선")
print("       - 0원 ~ 10만원: 최대 부스팅")
print("       - 10만원 이상: 부스팅 감소 (선형)")

# 프로파일 3: 고가 상품 우선 (프리미엄 전략)
profile_high_price = ScoringProfile(
    name="highPriceFirst",
    functions=[
        MagnitudeScoringFunction(
            field_name="normal_price",
            boost=5,
            parameters=MagnitudeScoringParameters(
                boosting_range_start=100000,
                boosting_range_end=500000,
                constant_boost_beyond_range=True
            ),
            interpolation=ScoringFunctionInterpolation.LINEAR
        )
    ]
)
print("\n   3️⃣  highPriceFirst: 고가 상품 우선")
print("       - 10만원 ~ 50만원: 선형 감소")
print("       - 50만원 이상: 최대 부스팅 유지")

# 프로파일 4: 프리미엄 브랜드 우선
profile_premium_brand = ScoringProfile(
    name="premiumBrandFirst",
    functions=[
        TagScoringFunction(
            field_name="brand",
            boost=10,
            parameters=TagScoringParameters(
                tags_parameter="premiumBrands"
            )
        )
    ]
)
print("\n   4️⃣  premiumBrandFirst: 특정 브랜드 우선")
print("       - 노스페이스, 라코스테 등 프리미엄 브랜드 10배 부스팅")

# 프로파일 5: 복합 프로파일
profile_combined = ScoringProfile(
    name="bestValue",
    text_weights=TextWeights(
        weights={
            "title": 2.0,
            "brand": 1.5
        }
    ),
    functions=[
        MagnitudeScoringFunction(
            field_name="normal_price",
            boost=3,
            parameters=MagnitudeScoringParameters(
                boosting_range_start=0,
                boosting_range_end=80000,
                constant_boost_beyond_range=False
            ),
            interpolation=ScoringFunctionInterpolation.QUADRATIC
        )
    ],
    function_aggregation="sum"
)
print("\n   5️⃣  bestValue: 최적 가성비 (복합)")
print("       - 필드 가중치 (title 2배, brand 1.5배)")
print("       - Magnitude: 0원 ~ 8만원에서 2차 함수 감소")
print("       - 결과 합산 방식")

print("\n✅ 5개의 Scoring Profile 정의 완료")

---
## 4️⃣ 인덱스에 Scoring Profile 추가

In [ ]:
# 기존 프로파일 보존, 새 프로파일 추가
existing_profiles = list(existing_index.scoring_profiles) if existing_index.scoring_profiles else []
existing_names = [p.name for p in existing_profiles]

new_profiles = [
    profile_field_weights,
    profile_low_price,
    profile_high_price,
    profile_premium_brand,
    profile_combined
]

print("\n📋 Scoring Profile 추가 처리...\n")
for new_profile in new_profiles:
    if new_profile.name not in existing_names:
        existing_profiles.append(new_profile)
        print(f"   ✅ '{new_profile.name}' 프로파일 추가")
    else:
        print(f"   ⚠️ '{new_profile.name}' 프로파일이 이미 존재 (스킵)")

# 인덱스 업데이트
existing_index.scoring_profiles = existing_profiles
result = index_client.create_or_update_index(existing_index)

print(f"\n✅ 인덱스 업데이트 완료")
print(f"   - 인덱스명: {result.name}")
print(f"   - Scoring Profiles 수: {len(result.scoring_profiles)}")
print(f"   - Scoring Profiles 목록:")
for profile in result.scoring_profiles:
    print(f"     • {profile.name}")

---
## 5️⃣ 데이터 상태 확인

In [ ]:
import pandas as pd

# CSV 데이터 확인
df = pd.read_csv("../00-data/sample_data.csv")

# 인덱스의 문서 개수 확인
results = search_client.search(search_text="*", top=1, include_total_count=True)
doc_count = results.get_count()

print(f"\n📊 데이터 상태 확인")
print(f"   - CSV 파일 행 수: {len(df)}")
print(f"   - 인덱스 문서 수: {doc_count}")

if doc_count > 0:
    print(f"\n✅ 기존 데이터 사용 (재업로드 불필요)")
else:
    print(f"\n⚠️ 인덱스가 비어있습니다. 데이터를 업로드해야 합니다.")
    print(f"   다음 폴더의 데이터 업로드 셀을 먼저 실행하세요:")
    print(f"   - 02-keyword_search/02-upload_data.ipynb")
    print(f"   - 또는 03-vector_search/02-upload_vectors.ipynb")

---
## 6️⃣ 검색 함수 정의

In [ ]:
def search_with_profile(query, profile_name=None, top=5):
    """Scoring Profile을 적용한 검색"""
    results = search_client.search(
        search_text=query,
        scoring_profile=profile_name,
        top=top,
        include_total_count=True
    )
    
    profile_text = profile_name if profile_name else "없음(기본)"
    print(f"\n🔍 검색어: '{query}'")
    print(f"📊 Scoring Profile: {profile_text}")
    print("=" * 80)
    
    results_list = []
    for idx, result in enumerate(results, 1):
        print(f"{idx}. {result['title'][:50]}...")
        print(f"   브랜드: {result['brand']} | 가격: {result['normal_price']:,}원")
        print(f"   점수: {result['@search.score']:.4f}")
        results_list.append({
            'title': result['title'],
            'brand': result['brand'],
            'price': result['normal_price'],
            'score': result['@search.score']
        })
    
    return results_list

print("✅ 검색 함수 정의 완료")

---
## 7️⃣ Scoring Profile 비교: 필드 가중치 (titleBoost)

In [ ]:
print("\n" + "="*80)
print(" 비교 1: 필드 가중치 효과 (titleBoost)")
print("="*80)
print("\n📌 시나리오: 검색어 '선물'")
print("   - 기본 검색: BM25 점수 기반 (필드 구분 없음)")
print("   - titleBoost: title 매칭 시 3배, brand 매칭 시 2배 부스팅")

# 기본 검색
print("\n\n📌 기본 검색 (프로파일 없음)")
results_default = search_with_profile("선물", profile_name=None, top=5)

# titleBoost 프로파일 적용
print("\n\n📌 titleBoost 프로파일 적용")
results_title = search_with_profile("선물", profile_name="titleBoost", top=5)

In [ ]:
# 점수 비교
print("\n\n📈 점수 비교")
print("-" * 80)
print(f"{'순위':<5} {'기본 점수':<15} {'titleBoost 점수':<15} {'상승도':<10} {'변화'}")
print("-" * 80)
for i, (d, t) in enumerate(zip(results_default, results_title), 1):
    diff = t['score'] - d['score']
    change_pct = ((t['score'] - d['score']) / d['score'] * 100) if d['score'] > 0 else 0
    diff_text = f"+{diff:.4f}" if diff > 0 else f"{diff:.4f}"
    change_indicator = "📈" if diff > 0 else "📉"
    print(f"{i:<5} {d['score']:<15.4f} {t['score']:<15.4f} {change_pct:>+6.1f}%    {change_indicator}")

---
## 8️⃣ Scoring Profile 비교: 가격 기반 부스팅

In [ ]:
print("\n" + "="*80)
print(" 비교 2: 가격 기반 부스팅 (저가 vs 고가)")
print("="*80)
print("\n📌 시나리오: 검색어 '자켓'")
print("   - lowPriceFirst: 0원 ~ 10만원에서 높은 부스팅")
print("   - highPriceFirst: 10만원 ~ 50만원에서 높은 부스팅")

# 저가 우선
print("\n\n💰 저가 상품 우선 (lowPriceFirst)")
results_low = search_with_profile("자켓", profile_name="lowPriceFirst", top=5)

# 고가 우선
print("\n\n💎 고가 상품 우선 (highPriceFirst)")
results_high = search_with_profile("자켓", profile_name="highPriceFirst", top=5)

In [ ]:
# 가격 순서 비교
print("\n\n💵 가격대 비교")
print("-" * 100)
print(f"{'#':<3} {'lowPriceFirst (가성비)':<35} {'highPriceFirst (프리미엄)':<35}")
print("-" * 100)
max_len = max(len(results_low), len(results_high))
for i in range(max_len):
    if i < len(results_low):
        low_title = results_low[i]['title'][:28]
        low_price = f"₩{results_low[i]['price']:,}"
    else:
        low_title = "-"
        low_price = "-"
    
    if i < len(results_high):
        high_title = results_high[i]['title'][:28]
        high_price = f"₩{results_high[i]['price']:,}"
    else:
        high_title = "-"
        high_price = "-"
    
    print(f"{i+1:<3} {low_title:<28} {low_price:<6} {high_title:<28} {high_price:<6}")

---
## 9️⃣ Scoring Profile 비교: Tag 함수 (프리미엄 브랜드)

In [ ]:
print("\n" + "="*80)
print(" 비교 3: 프리미엄 브랜드 부스팅 (Tag 함수)")
print("="*80)

premium_brands = ["노스페이스", "라코스테 우먼", "앤더슨벨"]

print(f"\n📌 시나리오: 검색어 '자켓'")
print(f"🏷️ 부스팅 대상 브랜드: {premium_brands}")
print(f"   - 위 브랜드 매칭 시 10배 부스팅 적용")

# 기본 검색
print("\n\n📌 기본 검색 (프로파일 없음)")
results_no_tag = search_with_profile("자켓", profile_name=None, top=5)

# Tag 함수 적용
print("\n\n📌 premiumBrandFirst 프로파일 (Tag 함수)")
results_with_tag = search_client.search(
    search_text="자켓",
    scoring_profile="premiumBrandFirst",
    scoring_parameters=[f"premiumBrands-{','.join(premium_brands)}"],
    top=5
)

print(f"\n🔍 검색어: '자켓'")
print(f"🏷️ 부스팅 브랜드: {premium_brands}")
print("=" * 80)

for idx, result in enumerate(results_with_tag, 1):
    is_premium = "⭐" if result['brand'] in premium_brands else "  "
    print(f"{idx}. {is_premium} {result['title'][:45]}...")
    print(f"      브랜드: {result['brand']} | 가격: {result['normal_price']:,}원")
    print(f"      점수: {result['@search.score']:.4f}")

---
## 🔟 Scoring Profile 비교: 복합 프로파일 (bestValue)

In [ ]:
print("\n" + "="*80)
print(" 비교 4: 복합 프로파일 (필드 가중치 + 저가 우선)")
print("="*80)
print("\n📌 시나리오: 검색어 '유아용품 선물'")
print("   - bestValue 프로파일:")
print("     1) title 필드 2배 부스팅")
print("     2) brand 필드 1.5배 부스팅")
print("     3) 0원 ~ 8만원 가격대에서 2차 함수로 부스팅")

# 기본 검색
print("\n\n📌 기본 검색 (프로파일 없음)")
results_basic = search_with_profile("유아용품 선물", profile_name=None, top=5)

# 복합 프로파일 적용
print("\n\n📌 bestValue 프로파일 (복합)")
results_best = search_with_profile("유아용품 선물", profile_name="bestValue", top=5)

In [ ]:
# 복합 프로파일 효과 분석
print("\n\n📊 복합 프로파일 효과 분석")
print("-" * 80)
print(f"{'#':<3} {'기본':<15} {'bestValue':<15} {'점수 상승':<15} {'특징'}")
print("-" * 80)
for i, (b, bv) in enumerate(zip(results_basic, results_best), 1):
    diff = bv['score'] - b['score']
    # 특징 분석
    feature = ""
    if "선물" in bv['title'] or "세트" in bv['title']:
        feature = "선물/세트"
    if bv['price'] < 50000:
        feature += " 저가" if feature else "저가"
    
    diff_text = f"+{diff:.4f}" if diff > 0 else f"{diff:.4f}"
    print(f"{i:<3} {b['score']:<15.4f} {bv['score']:<15.4f} {diff_text:<15} {feature}")

---
## 1️⃣1️⃣ Interpolation 방식 비교

Magnitude 함수에서 점수가 어떻게 감소하는지 보간 방식을 시각화합니다.

In [ ]:
import math

# Interpolation 방식별 점수 계산
def calculate_boost(value, start, end, boost, interpolation):
    """부스팅 점수 계산"""
    if value <= start:
        return boost
    elif value >= end:
        return 1
    
    position = (value - start) / (end - start)
    
    if interpolation == "linear":
        return boost - (boost - 1) * position
    elif interpolation == "quadratic":
        return boost - (boost - 1) * (position ** 2)
    elif interpolation == "logarithmic":
        return boost - (boost - 1) * math.log10(1 + 9 * position)
    elif interpolation == "constant":
        return boost

print("\n" + "="*80)
print(" Interpolation 방식별 부스팅 점수")
print("="*80)
print("\n가격 범위: 0원 ~ 100,000원, 부스팅 배수: 5")
print("-" * 75)
print(f"{'가격':<12} {'Linear':<12} {'Quadratic':<12} {'Logarithmic':<12} {'Constant'}")
print("-" * 75)

prices = [0, 20000, 40000, 60000, 80000, 100000, 150000]
for price in prices:
    linear = calculate_boost(price, 0, 100000, 5, "linear")
    quadratic = calculate_boost(price, 0, 100000, 5, "quadratic")
    logarithmic = calculate_boost(price, 0, 100000, 5, "logarithmic")
    constant = calculate_boost(price, 0, 100000, 5, "constant")
    
    print(f"{price:>10,}원 {linear:<12.2f} {quadratic:<12.2f} {logarithmic:<12.2f} {constant:.2f}")

### 📊 Interpolation 방식 설명

| 방식 | 특징 | 사용 케이스 |
|------|------|------------|
| **Linear** | 균일하게 감소 | 일반적인 가격 부스팅 |
| **Quadratic** | 처음에는 천천히, 끝에서 급격히 감소 | 저가 구간 강조 (2차 함수) |
| **Logarithmic** | 처음에 급격히, 끝에서 천천히 감소 | 중간 가격대 강조 |
| **Constant** | 범위 내 동일 부스팅 | 특정 범위 전체 강조 |

---
## 1️⃣2️⃣ 실무 시나리오: 카테고리별 맞춤 검색

In [ ]:
print("\n" + "="*80)
print(" 실무 시나리오 1: 유아동 - 가성비 상품 추천")
print("="*80)
print("\n📍 상황: 예산이 한정된 부모님을 위한 유아동 용품 추천")
print("📋 전략: lowPriceFirst 프로파일 + 유아동 카테고리 필터")

results = search_client.search(
    search_text="선물",
    filter="category eq '유아동'",
    scoring_profile="lowPriceFirst",
    top=5
)

print("\n🔍 검색 조건:")
print("   - 검색어: '선물'")
print("   - 필터: category = '유아동'")
print("   - Scoring Profile: lowPriceFirst")
print("\n" + "="*80)

for idx, result in enumerate(results, 1):
    print(f"\n{idx}. {result['title'][:50]}...")
    print(f"   💰 가격: {result['normal_price']:,}원 (가성비 추천!)")
    print(f"   브랜드: {result['brand']} | 점수: {result['@search.score']:.4f}")

In [ ]:
print("\n" + "="*80)
print(" 실무 시나리오 2: 스포츠 - 프리미엄 브랜드 강조")
print("="*80)
print("\n📍 상황: 운동용품 매장에서 프리미엄 브랜드 판매 촉진")
print("📋 전략: premiumBrandFirst 프로파일 + 스포츠/레져 필터")

sports_premium = ["노스페이스", "파타고니아", "젝시믹스"]

results = search_client.search(
    search_text="자켓 바람막이",
    filter="category eq '스포츠/레져'",
    scoring_profile="premiumBrandFirst",
    scoring_parameters=[f"premiumBrands-{','.join(sports_premium)}"],
    top=5
)

print(f"\n🔍 검색 조건:")
print(f"   - 검색어: '자켓 바람막이'")
print(f"   - 필터: category = '스포츠/레져'")
print(f"   - Scoring Profile: premiumBrandFirst")
print(f"   - 프리미엄 브랜드: {sports_premium}")
print("\n" + "="*80)

for idx, result in enumerate(results, 1):
    is_premium = "⭐ " if result['brand'] in sports_premium else "   "
    print(f"\n{idx}. {is_premium}{result['title'][:45]}...")
    print(f"   브랜드: {result['brand']} | 가격: {result['normal_price']:,}원")
    print(f"   점수: {result['@search.score']:.4f} {'(프리미엄 브랜드!)' if result['brand'] in sports_premium else ''}")

---
## 1️⃣3️⃣ 핵심 정리

### ✅ Scoring Profile의 장점

1. **비즈니스 요구사항 반영**
   - 단순 관련성이 아닌 비즈니스 목표에 맞춰 검색 결과 순서 조정
   - 예: 가성비, 프리미엄, 신제품 등

2. **필드별 차등 가중치**
   - 상품명(title) 매칭을 더 중요하게 평가
   - 리뷰 키워드는 보조 역할

3. **숫자 필드 활용**
   - 가격, 평점, 판매량 등으로 부스팅
   - 범위 설정 가능

4. **조건부 부스팅**
   - 특정 브랜드, 카테고리만 강조
   - 파트너사 상품 노출 등

### ⚠️ 주의사항

- **키워드 검색(BM25)에만 적용**: 벡터 검색, 하이브리드 검색의 RRF 점수에는 영향 없음
- **인덱스 업데이트 필요**: 프로파일 변경 시 인덱스를 다시 생성/업데이트해야 함
- **테스트 필수**: 실제 검색 결과를 확인하여 최적 파라미터 찾기

### 🎯 실무 활용 가이드

| 비즈니스 목표 | 권장 프로파일 | 파라미터 |
|--------------|---------------|----------|
| **가성비 상품 추천** | Magnitude (저가) | 0 ~ 10만원에서 높은 부스팅 |
| **프리미엄 상품 판매** | Magnitude (고가) | 10만원 이상에서 높은 부스팅 |
| **정확도 향상** | Text Weights | title 3배, brand 2배 |
| **파트너 상품 노출** | Tag | 파트너 브랜드 10배 부스팅 |
| **최적 균형** | Combined | 필드 가중치 + Magnitude 조합 |

In [ ]:
print("\n✅ Scoring Profile 실습 완료!")
print("\n📚 다음 학습:")
print("   - 06-re_ranking: Semantic Ranker로 AI 기반 재순위화")
print("   - 07-enriched_dataset: AI Skills로 데이터 강화")
print("   - 08-ai-skills: 커스텀 AI 능력 구현")